# Recollida de dades

Aquest notebook genera la mostra inicial del treball: 1000 colors RGB aleatoris, el seu chroma i les imatges PNG que despres passarem als models.

La logica important esta a `scripts/utils.py`. Aqui intento deixar nomes la configuracio i les crides principals, perque si no el notebook es fa impossible de seguir.

## Configuracio

Fixem els parametres de l'experiment. La seed es important per poder regenerar exactament la mateixa mostra en un altre ordinador.

In [ ]:
from pathlib import Path
import sys

N_COLORS = 1000
SEED = 23
IMAGE_SIZE = 100
NEAR_DISTANCE = 30

base_dir = Path("recollida-dades") if Path("recollida-dades").exists() else Path(".")
csv_dir = base_dir / "csv"
images_dir = base_dir / "images"
logs_dir = base_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)

## Carrega de scripts

Importem les funcions del script. Si alguna no existeix, parem aqui, perque probablement el notebook no esta trobant be `scripts/utils.py`.

In [ ]:
from utils import (
    build_final_sample,
    collect_model_outputs,
    generate_images_from_sample,
    generate_unique_colors,
    save_csv,
    write_log,
)

required_functions = [
    "generate_unique_colors",
    "generate_images_from_sample",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no cargados correctamente: {missing}")

write_log("Notebook de recollida inicialitzat", logs_dir / "pipeline.log")

## Generacio de la mostra de colors

Generem 1000 colors unics de la paleta RGB. Guardem `R`, `G`, `B`, el codi hexadecimal i el chroma calculat amb CIELAB.

Aixo es el fitxer d'entrada que despres ens permet saber quin color era el correcte per cada imatge.

In [ ]:
input_sample = generate_unique_colors(n=N_COLORS, seed=SEED)
input_path = csv_dir / "input_image_sample.csv"

save_csv(input_sample, input_path)
write_log(f"Mostra generada: {input_path}", logs_dir / "pipeline.log")

input_sample.head()

## Comprovacio rapida de biaix

Fem una mirada rapida als canals RGB. No es una prova estadistica formal, pero ajuda a detectar si alguna cosa ha sortit molt malament.

In [ ]:
input_sample[["r", "g", "b", "chroma"]].describe()

## Generacio de les imatges

Cada imatge es de 100x100 pixels: 60% color objectiu, 20% color proper i 20% color aleatori. Les guardem en PNG per no modificar el color amb compressio JPEG.

El nom de cada fitxer es el color hexadecimal, per exemple `C81619.png`.

In [ ]:
image_paths = generate_images_from_sample(
    input_sample,
    output_dir=images_dir,
    size=IMAGE_SIZE,
    near_distance=NEAR_DISTANCE,
    seed=SEED,
)

write_log(f"Imatges generades: {len(image_paths)}", logs_dir / "pipeline.log")
len(image_paths), image_paths[:3]

## Consulta als models

Aquesta part queda preparada pero no l'executo automaticament. Si la llancem sense voler, gastarem crides d'API.

Quan tinguem la clau configurada, la idea es generar `outputmodel_image_sample.csv` amb una fila per imatge i model.

In [ ]:
# from openai import OpenAI
#
# client = OpenAI()
# models = ["gpt-4o", "gpt-4o-mini"]
# model_outputs = collect_model_outputs(
#     client=client,
#     image_paths=image_paths,
#     models=models,
#     temperature=0.2,
# )
#
# output_path = csv_dir / "outputmodel_image_sample.csv"
# save_csv(model_outputs, output_path)
# write_log(f"Resultats de models guardats: {output_path}", logs_dir / "pipeline.log")

## Dataset final per analisi

Quan existeixi el CSV de sortida dels models, unim l'entrada i la sortida i calculem l'error cromatic en RGB. Aquest fitxer sera el que carregarem a R.

In [ ]:
# output_path = csv_dir / "outputmodel_image_sample.csv"
# final_path = csv_dir / "sample-colors.csv"
#
# if output_path.exists():
#     model_outputs = pd.read_csv(output_path)
#     final_sample = build_final_sample(input_sample, model_outputs)
#     save_csv(final_sample, final_path)
#     final_sample.head()
# else:
#     print("Encara no existeix outputmodel_image_sample.csv")

## Resultats generats

Comprovem que tenim el CSV d'entrada i les imatges. Si aqui surt 1000, la part base de recollida ja esta feta.

In [ ]:
print("CSV generats:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges:", len(list(images_dir.glob("*.png"))))
print("Logs:", sorted(path.name for path in logs_dir.iterdir()))